# Russia HS6 Statistical Analysis

입력: `Russia_BACI_Lite_Dataset.zip`

원본 1.2GB BACI는 더 이상 읽지 않습니다.

분석:
- HS6 pre/post 변화량
- Wilcoxon signed-rank test
- Bootstrap 95% CI
- Sign-flip permutation test
- Rank-biserial effect size
- Non-energy robustness
- China vs EU share-change correlation
- HS6 고의존 품목 수 추세


In [ ]:
!pip -q install pandas numpy scipy matplotlib pyarrow openpyxl

from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import zipfile, shutil

ROOT = Path("/content/russia_hs6_stats")
DATA = ROOT/"data"
OUT = ROOT/"output"
FIG = OUT/"figures"
TAB = OUT/"tables"
for p in [DATA,OUT,FIG,TAB]:
    p.mkdir(parents=True,exist_ok=True)

SEARCH_DIRS = [Path("/content"),Path("/content/BACI"),Path("/content/drive/MyDrive/BACI")]

def find_zip():
    for d in SEARCH_DIRS:
        if d.exists():
            z = d/"Russia_BACI_Lite_Dataset.zip"
            if z.exists():
                return z
    try:
        from google.colab import drive
        drive.mount("/content/drive",force_remount=False)
    except Exception:
        pass
    for d in SEARCH_DIRS:
        z = d/"Russia_BACI_Lite_Dataset.zip"
        if z.exists():
            return z
    return None

LITE_ZIP = find_zip()
if LITE_ZIP is None:
    raise FileNotFoundError("Russia_BACI_Lite_Dataset.zip을 /content 또는 BACI 폴더에 두세요.")

print("Lite dataset:",LITE_ZIP)


In [ ]:
if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)

with zipfile.ZipFile(LITE_ZIP) as z:
    z.extractall(DATA)

imp = pd.read_parquet(DATA/"russia_import_hs6_year_summary.parquet")
exp = pd.read_parquet(DATA/"russia_export_hs6_year_summary.parquet")

PRE=[2019,2020,2021]
POST=[2023,2024]

print("Import",imp.shape,"Export",exp.shape)


In [ ]:
def build_prepost(df, cols):
    pre = df[df.year.isin(PRE)].groupby(["hs6","hs2"],as_index=False)[cols].mean()
    post = df[df.year.isin(POST)].groupby(["hs6","hs2"],as_index=False)[cols].mean()
    pre = pre.rename(columns={c:f"{c}_pre" for c in cols})
    post = post.rename(columns={c:f"{c}_post" for c in cols})
    out = pre.merge(post,on=["hs6","hs2"],how="inner")
    for c in cols:
        out[f"delta_{c}"] = out[f"{c}_post"]-out[f"{c}_pre"]
    return out

cols=["china_share","eu27_share","india_share","turkiye_share"]
imp_pp=build_prepost(imp,cols)
exp_pp=build_prepost(exp,cols)

imp_pp.to_csv(TAB/"HS6_import_prepost_changes.csv",index=False)
exp_pp.to_csv(TAB/"HS6_export_prepost_changes.csv",index=False)
display(imp_pp.head())


In [ ]:
def rank_biserial(d):
    d=np.asarray(d)
    d=d[np.isfinite(d)&(d!=0)]
    ranks=stats.rankdata(np.abs(d))
    wp=ranks[d>0].sum(); wm=ranks[d<0].sum()
    return (wp-wm)/(wp+wm) if (wp+wm)>0 else np.nan

def bootstrap_ci(d,func=np.mean,B=5000,seed=42):
    d=np.asarray(d); d=d[np.isfinite(d)]
    rng=np.random.default_rng(seed)
    vals=np.array([func(rng.choice(d,size=len(d),replace=True)) for _ in range(B)])
    return np.quantile(vals,[.025,.975])

def signflip_test(d,B=10000,seed=42):
    d=np.asarray(d); d=d[np.isfinite(d)]
    obs=np.mean(d)
    rng=np.random.default_rng(seed)
    sims=np.array([(d*rng.choice([-1,1],size=len(d))).mean() for _ in range(B)])
    p=(np.sum(np.abs(sims)>=abs(obs))+1)/(B+1)
    return obs,p

def summarize(d,label):
    d=np.asarray(d); d=d[np.isfinite(d)]
    wil=stats.wilcoxon(d,zero_method="wilcox",alternative="two-sided",mode="auto")
    mci=bootstrap_ci(d,np.mean)
    medci=bootstrap_ci(d,np.median)
    _,pp=signflip_test(d)
    return {
        "outcome":label,
        "n_hs6":len(d),
        "mean_change_pp":100*np.mean(d),
        "median_change_pp":100*np.median(d),
        "share_positive_pct":100*np.mean(d>0),
        "wilcoxon_p":wil.pvalue,
        "bootstrap_mean_lo_pp":100*mci[0],
        "bootstrap_mean_hi_pp":100*mci[1],
        "bootstrap_median_lo_pp":100*medci[0],
        "bootstrap_median_hi_pp":100*medci[1],
        "permutation_p":pp,
        "rank_biserial":rank_biserial(d)
    }

rows=[]
for df,name in [(imp_pp,"Import"),(exp_pp,"Export")]:
    for c,label in [
        ("delta_china_share",f"{name}: China share"),
        ("delta_eu27_share",f"{name}: EU27 share"),
        ("delta_india_share",f"{name}: India share"),
        ("delta_turkiye_share",f"{name}: Turkiye share")
    ]:
        rows.append(summarize(df[c],label))

stats_table=pd.DataFrame(rows)
stats_table.to_csv(TAB/"HS6_statistical_tests.csv",index=False)
display(stats_table.round(4))


In [ ]:
nonenergy=pd.DataFrame([
    summarize(imp_pp[imp_pp.hs2.ne("27")]["delta_china_share"],"Import non-energy: China"),
    summarize(exp_pp[exp_pp.hs2.ne("27")]["delta_china_share"],"Export non-energy: China"),
    summarize(imp_pp[imp_pp.hs2.ne("27")]["delta_eu27_share"],"Import non-energy: EU27"),
    summarize(exp_pp[exp_pp.hs2.ne("27")]["delta_eu27_share"],"Export non-energy: EU27")
])
nonenergy.to_csv(TAB/"HS6_nonenergy_tests.csv",index=False)
display(nonenergy.round(4))


In [ ]:
d=imp_pp[["hs6","hs2","delta_china_share","delta_eu27_share"]].dropna()
pearson=stats.pearsonr(d["delta_eu27_share"],d["delta_china_share"])
spearman=stats.spearmanr(d["delta_eu27_share"],d["delta_china_share"])

corr=pd.DataFrame([{
    "n":len(d),
    "pearson_r":pearson.statistic,
    "pearson_p":pearson.pvalue,
    "spearman_rho":spearman.statistic,
    "spearman_p":spearman.pvalue,
    "share_EUdown_ChinaUp_pct":
        100*np.mean((d["delta_eu27_share"]<0)&(d["delta_china_share"]>0))
}])
corr.to_csv(TAB/"China_EU_change_correlation.csv",index=False)
display(corr.round(4))

plt.figure(figsize=(7.5,6))
plt.scatter(100*d["delta_eu27_share"],100*d["delta_china_share"],alpha=.35,s=16)
plt.axhline(0,linewidth=1); plt.axvline(0,linewidth=1)
plt.xlabel("Change in EU27 share (percentage points)")
plt.ylabel("Change in China share (percentage points)")
plt.title("HS6-level import reorientation: EU27 vs China")
plt.tight_layout()
plt.savefig(FIG/"HS6_China_vs_EU_scatter.png",dpi=180)
plt.show()


In [ ]:
delta=imp_pp["delta_china_share"].dropna()
plt.figure(figsize=(8.5,5.5))
plt.hist(100*delta,bins=50)
plt.axvline(0,linewidth=1)
plt.axvline(100*delta.mean(),linestyle="--",linewidth=1)
plt.xlabel("Post − pre China share (percentage points)")
plt.ylabel("Number of HS6 products")
plt.title("Distribution of HS6-level changes in China's import share")
plt.tight_layout()
plt.savefig(FIG/"HS6_import_China_share_change_distribution.png",dpi=180)
plt.show()


In [ ]:
dep_rows=[]
for y,g in imp.groupby("year"):
    dep_rows.append({
        "year":y,
        "eligible":int(g["china_share"].notna().sum()),
        "china_ge_50":int((g["china_share"]>=.5).sum()),
        "china_ge_70":int((g["china_share"]>=.7).sum()),
        "china_ge_90":int((g["china_share"]>=.9).sum())
    })
dep=pd.DataFrame(dep_rows)
dep.to_csv(TAB/"HS6_high_dependence_counts_by_year.csv",index=False)
display(dep)

plt.figure(figsize=(9,5.5))
plt.plot(dep.year,dep.china_ge_50,marker="o",label="China share ≥50%")
plt.plot(dep.year,dep.china_ge_70,marker="o",label="China share ≥70%")
plt.plot(dep.year,dep.china_ge_90,marker="o",label="China share ≥90%")
plt.axvline(2022,linestyle="--",linewidth=1)
plt.xlabel("Year"); plt.ylabel("Number of HS6 products")
plt.title("Russia import products with high dependence on China")
plt.legend(); plt.tight_layout()
plt.savefig(FIG/"HS6_high_dependence_counts.png",dpi=180)
plt.show()


In [ ]:
xlsx=OUT/"Russia_HS6_Statistical_Results.xlsx"
with pd.ExcelWriter(xlsx,engine="openpyxl") as w:
    imp_pp.to_excel(w,sheet_name="Import_prepost",index=False)
    exp_pp.to_excel(w,sheet_name="Export_prepost",index=False)
    stats_table.to_excel(w,sheet_name="Statistical_tests",index=False)
    nonenergy.to_excel(w,sheet_name="Nonenergy_tests",index=False)
    corr.to_excel(w,sheet_name="China_EU_corr",index=False)
    dep.to_excel(w,sheet_name="Dependence_counts",index=False)

zip_out=ROOT/"Russia_HS6_Statistical_Outputs.zip"
if zip_out.exists(): zip_out.unlink()
shutil.make_archive(str(zip_out.with_suffix("")),"zip",OUT)

print("Excel:",xlsx)
print("ZIP:",zip_out)

try:
    from google.colab import files
    files.download(str(zip_out))
except Exception:
    pass
